# DeepSafety API Endpoints Tutorial Workbook

This single notebook is a practical, realistic walkthrough of the DeepSafety API surface.

It is intentionally scenario-driven and grouped by engineering workflow.


## 1) Setup

This notebook uses FastAPI's `TestClient`, so users can run everything in Jupyter without starting an external server process.


In [ ]:
from pprint import pprint

from fastapi.testclient import TestClient

from deepsafety.api import create_app

client = TestClient(create_app())

def call(method: str, path: str, payload: dict | None = None) -> dict:
    if method.upper() == 'GET':
        response = client.get(path)
    else:
        response = client.post(path, json=payload)
    if response.status_code >= 400:
        raise RuntimeError(f"{method} {path} failed: {response.status_code} -> {response.text}")
    return response.json()

print('DeepSafety notebook client ready')


## 2) Foundational Material Records

In [ ]:
materials = call('GET', '/materials?q=ammonia&page=1&pageSize=5')
ammonia = call('GET', '/materials/ammonia')
toxicity = call('GET', '/materials/ammonia/toxicity')
flammability = call('GET', '/materials/ammonia/flammability')
reactivity = call('GET', '/materials/ammonia/reactivity')

print('Material IDs from query:')
print([item['id'] for item in materials['items']])
print('\nAmmonia key properties:')
pprint({
    'name': ammonia.get('name'),
    'molecularWeight': ammonia.get('molecularWeight'),
    'density': ammonia.get('density'),
    'boilingPoint': ammonia.get('boilingPoint'),
    'toxicity': toxicity,
    'flammability': flammability,
    'reactivity': reactivity,
})


## 3) Scenario Definition And Scenario-Library Templates

In [ ]:
scenario_realistic = call('POST', '/scenario-engine/define', {
    'incident_type': 'pipe_rupture',
    'classification': 'realistic_case',
    'inventory': {'phase': 'gas', 'mass_kg': 2000},
    'equipment': {'type': 'pipe', 'diameter_m': 0.2},
    'meteorology': {'wind_speed_m_s': 3.2, 'stability_class': 'D'},
    'topography': 'urban'
})

scenario_worst = call('POST', '/scenario-engine/define', {
    'incident_type': 'tank_leak',
    'classification': 'worst_case',
    'inventory': {'phase': 'liquid', 'mass_kg': 8000},
    'equipment': {'type': 'tank', 'diameter_m': 8.0},
})

templates = call('GET', '/scenario-library/templates')
selected = call('POST', '/source-models/scenario/select', {
    'scenarioType': 'worst_case',
    'equipmentType': 'tank',
    'inventoryMass': 8000,
    'siteTopography': 'urban'
})

conservative = call('POST', '/source-models/conservative-analysis', {
    'baseCase': {
        'release_duration_s': 240,
        'wind_speed_m_s': 3.0,
        'stability_class': 'D',
        'inventory_mass_kg': 600,
        'hole_diameter_m': 0.01
    },
    'maximize': ['release_duration', 'wind_speed', 'stability', 'inventory', 'hole_size']
})

print('Realistic assumption set:', scenario_realistic['scenario']['assumption_set'])
print('Worst-case assumption set:', scenario_worst['scenario']['assumption_set'])
print('Template IDs:', [t['id'] for t in templates])
print('Selected release scenario assumptions:')
pprint(selected['assumptions'])
print('Conservative-case rationale:')
pprint(conservative['rationale'])


## 4) Source Models: Gas/Vapor, Liquid, Flashing, Pool, Evaporation

In [ ]:
liquid_hole = call('POST', '/source-models/liquid-hole', {
    'liquidDensity': 820,
    'upstreamPressure': 700000,
    'downstreamPressure': 101325,
    'holeArea': 0.000314,
})
liquid_pipe = call('POST', '/source-models/liquid-pipe', {
    'density': 820,
    'pipeDiameter': 0.05,
    'pressureDrop': 300000,
    'pipeLength': 80,
    'viscosity': 0.001,
    'roughness': 0.000045,
})
gas_hole = call('POST', '/source-models/gas-hole', {
    'upstreamPressure': 6_000_000,
    'downstreamPressure': 101_325,
    'temperatureK': 300,
    'molecularWeight': 16,
    'heatCapacityRatio': 1.3,
    'holeArea': 0.0000785,
})
gas_pipe = call('POST', '/source-models/gas-pipe', {
    'pipeDiameter': 0.1,
    'pipeLength': 150,
    'roughness': 0.00015,
    'upstreamPressure': 6_000_000,
    'downstreamPressure': 101_325,
    'temperatureK': 300,
    'molecularWeight': 16,
    'heatCapacityRatio': 1.3,
})
flashing = call('POST', '/source-models/flashing-liquid', {
    'materialId': 'ammonia',
    'initialTemperatureK': 320,
    'inventoryMass': 500,
    'cpLiquid': 2200,
    'latentHeat': 1300000,
})
pool = call('POST', '/source-models/solve', {
    'model_type': 'pool_formation',
    'inputs': {
        'liquid_mass_kg': 1000,
        'density_kg_m3': 820,
        'pool_thickness_m': 0.01,
        'containment_area_m2': 150,
    },
})
evap = call('POST', '/source-models/solve', {
    'model_type': 'evaporation',
    'inputs': {
        'area_m2': pool['outputs']['pool_area_m2'],
        'latent_heat_j_kg': 1_300_000,
        'heat_flux_kw_m2': 1.5,
    },
})

pprint({
    'liquid_hole_release_rate_kg_s': liquid_hole['releaseRate'],
    'liquid_pipe_release_rate_kg_s': liquid_pipe['massFlowRate'],
    'gas_hole_mass_flow_kg_s': gas_hole['massFlowRate'],
    'gas_pipe_mass_flow_kg_s': gas_pipe['massFlowRate'],
    'flash_vapor_fraction': flashing['vaporFraction'],
    'pool_area_m2': pool['outputs']['pool_area_m2'],
    'evap_rate_kg_s': evap['outputs']['evaporation_rate_kg_s'],
})


## 5) Dispersion: Gaussian Plume/Puff, Dense Gas, Isopleths, Toxic Endpoints, Mitigation

In [ ]:
plume = call('POST', '/dispersion/gaussian-plume', {
    'releaseRate': gas_hole['massFlowRate'],
    'releaseHeight': 1.5,
    'windSpeed': 3.0,
    'stabilityClass': 'D',
    'receptorGrid': {'xMin': 50, 'xMax': 300, 'yMin': -100, 'yMax': 100, 'z': 1.5, 'dx': 50, 'dy': 50},
})
puff = call('POST', '/dispersion/gaussian-puff', {
    'releasedMass': gas_hole['massFlowRate'] * 60,
    'windSpeed': 2.5,
    'stabilityClass': 'E',
    'receptorGrid': {'xMin': 50, 'xMax': 300, 'yMin': -100, 'yMax': 100, 'z': 1.5, 'dx': 50, 'dy': 50},
})
dense = call('POST', '/dispersion/dense-gas', {
    'releasedMass': 500,
    'gasDensityKgM3': 2.5,
    'releaseDurationS': 180,
    'windSpeed': 2.0,
    'receptorGrid': {'xMin': 0, 'xMax': 250, 'yMin': -80, 'yMax': 80, 'z': 1.0, 'dx': 50, 'dy': 40},
})
isopleth = call('POST', '/dispersion/isopleth', {
    'dispersionResultId': plume['resultId'],
    'threshold': 0.0002,
})
toxic_endpoints = call('POST', '/dispersion/toxic-endpoints/evaluate', {
    'dispersionResultId': plume['resultId'],
    'criteria': [{'name': 'AEGL-2', 'value': 0.00015}, {'name': 'ERPG-2', 'value': 0.00025}],
})
mitigated = call('POST', '/dispersion/prevention-mitigation', {
    'releaseRate': gas_hole['massFlowRate'],
    'mitigationFactor': 0.45,
    'windSpeed': 3.0,
})

print('Plume max concentration:', plume['maxConcentration'])
print('Puff max concentration:', puff['maxConcentration'])
print('Dense-gas max concentration:', dense['maxConcentration'])
print('Isopleth max distance:', isopleth['maxDistance'])
pprint(toxic_endpoints['criteriaResults'])
pprint(mitigated)


## 6) Fire And Explosion Models

In [ ]:
flammability_mix = call('POST', '/fire-explosion/flammability/mixture', {
    'components': [{'materialId': 'methane', 'moleFraction': 0.6}, {'materialId': 'propane', 'moleFraction': 0.4}],
    'oxygenFraction': 0.21,
})
loc = call('POST', '/fire-explosion/loc', {'fuelSystem': {'materialId': 'propane'}, 'inertGas': 'nitrogen', 'operatingOxygenPercent': 10.5})
ignition = call('POST', '/fire-explosion/ignition-energy', {'minimumIgnitionEnergy': 0.25, 'availableIgnitionEnergy': 0.4})
tnt_eq = call('POST', '/fire-explosion/tnt-equivalency', {'chemicalEnergy': 900000, 'efficiency': 0.08})
multi_energy = call('POST', '/fire-explosion/multi-energy', {'tntEquivalentMass': tnt_eq['tntEquivalentMass'], 'distances': [25, 50, 100, 200]})
vce = call('POST', '/fire-explosion/vce', {'releasedMass': 400, 'vaporizedFraction': 0.35, 'congestionLevel': 'high', 'delayedIgnition': True, 'heatOfCombustion': 46000})
bleve = call('POST', '/fire-explosion/bleve', {'inventoryMass': 1200, 'liquidTemperatureK': 330, 'atmosphericBoilingPointK': 231, 'flammable': True, 'toxic': False})
jet_fire = call('POST', '/fire-explosion-models/solve', {'model_type': 'jet_fire', 'inputs': {'release_rate_kg_s': 2.5, 'heat_of_combustion_kj_kg': 46000, 'distance_m': 35}})
pool_fire = call('POST', '/fire-explosion-models/solve', {'model_type': 'pool_fire', 'inputs': {'pool_area_m2': 180, 'burning_flux_kg_m2_s': 0.05, 'heat_of_combustion_kj_kg': 44000, 'distance_m': 50}})
fireball = call('POST', '/fire-explosion-models/solve', {'model_type': 'fireball_bleve', 'inputs': {'fuel_mass_kg': 1500, 'heat_of_combustion_kj_kg': 46000, 'distance_m': 80}})
deflagration = call('POST', '/fire-explosion-models/solve', {'model_type': 'deflagration_screening', 'inputs': {'cloud_mass_kg': 250, 'heat_of_combustion_kj_kg': 46000, 'flame_speed_m_s': 180, 'confinement_factor': 1.3, 'distance_m': 100}})
detonation = call('POST', '/fire-explosion-models/solve', {'model_type': 'detonation_screening', 'inputs': {'cloud_mass_kg': 180, 'heat_of_combustion_kj_kg': 46000, 'detonable_fraction': 0.28, 'distance_m': 120}})
blast_damage = call('POST', '/fire-explosion-models/solve', {'model_type': 'blast_damage_screening', 'inputs': {'overpressure_kpa': 42, 'impulse_kpa_s': 18}})
mitigation_screen = call('POST', '/fire-explosion-models/solve', {'model_type': 'mitigation_screening', 'inputs': {'overpressure_kpa': 42, 'barrier_efficiency': 0.35, 'venting_factor': 0.4, 'target_overpressure_kpa': 14}})

pprint({
    'flammable_mixture': flammability_mix['flammable'],
    'loc_margin': loc['margin'],
    'can_ignite': ignition['canIgnite'],
    'vce_severity': vce['qualitativeSeverity'],
    'bleve_fireball_possible': bleve['fireballPossible'],
    'jet_fire_heat_flux': jet_fire['outputs']['heat_flux_kw_m2'],
    'pool_fire_heat_flux': pool_fire['outputs']['heat_flux_kw_m2'],
    'fireball_duration_s': fireball['outputs']['fireball_duration_s'],
    'deflagration_overpressure': deflagration['outputs']['overpressure_kpa'],
    'detonation_overpressure': detonation['outputs']['overpressure_kpa'],
    'blast_damage_category': blast_damage['outputs']['structural_damage'],
    'mitigated_overpressure': mitigation_screen['outputs']['mitigated_overpressure_kpa'],
})


## 7) Health And Industrial Hygiene Utilities

In [ ]:
convert = call('POST', '/health/convert-concentration', {
    'value': 150,
    'fromUnit': 'ppm',
    'toUnit': 'mg/m3',
    'temperatureK': 298.15,
    'pressureAtm': 1.0,
    'molecularWeight': 17.031,
})
probit = call('POST', '/health/probit/evaluate', {'k1': -14.3, 'k2': 2.3, 'variableValue': 2500})
twa = call('POST', '/health/exposure/twa', {
    'segments': [
        {'concentration': 45, 'durationMinutes': 120, 'concentrationUnit': 'ppm'},
        {'concentration': 15, 'durationMinutes': 360, 'concentrationUnit': 'ppm'},
    ]
})
compliance = call('POST', '/health/exposure/compliance', {
    'exposureProfile': {'segments': [{'concentration': 45, 'durationMinutes': 120}, {'concentration': 15, 'durationMinutes': 360}]},
    'criteria': {'tlv_twa': {'value': 25, 'unit': 'ppm'}, 'pel': {'value': 50, 'unit': 'ppm'}},
})
dilution = call('POST', '/industrial-hygiene/ventilation/dilution', {
    'generationRate': 250,
    'targetConcentration': 25,
    'targetUnit': 'ppm',
    'mixingFactor': 1.2,
    'roomConditions': {'temperatureK': 298.15, 'pressureAtm': 1.0, 'molecularWeight': 17.031},
})
local_exhaust = call('POST', '/industrial-hygiene/ventilation/local-exhaust', {'captureVelocity': 0.6, 'hoodArea': 1.8, 'losses': 0.25})
pool_evap = call('POST', '/industrial-hygiene/liquid-pool/evaporation', {
    'materialId': 'ammonia',
    'poolArea': 120,
    'ambient': {'temperatureK': 303.15, 'pressureAtm': 1.0, 'windSpeed': 2.8},
})

pprint({
    'converted_mg_m3': convert['outputValue'],
    'probit_probability': probit['probability'],
    'twa_ppm': twa['twa'],
    'compliance_results': compliance['results'],
    'dilution_ventilation_m3_s': dilution['requiredVentilationRate'],
    'local_exhaust_m3_s': local_exhaust['volumetricFlowRate'],
    'pool_evap_kg_s': pool_evap['evaporationRate'],
})


## 8) Prevention, Reactivity, And Relief-System Endpoints

In [ ]:
purge = call('POST', '/prevention/inerting/purge', {'method': 'pressure_vacuum_purging', 'vesselVolume': 40, 'initialOxygenPercent': 21, 'targetOxygenPercent': 6, 'purgeGasPurity': 0.995})
static_risk = call('POST', '/prevention/static-electricity/risk', {'conductivity': 5e-9, 'flowRate': 1.6, 'groundingBondingPresent': False, 'flammableAtmospherePresent': True})
area = call('POST', '/prevention/area-classification', {'zoneType': 'secondary', 'ventilationQuality': 'fair', 'materialId': 'propane'})
fire_protection = call('POST', '/prevention/fire-protection/strategy', {'hazards': ['pool_fire', 'vce'], 'activeSystems': ['deluge'], 'passiveSystems': ['fireproofing'], 'infrastructure': ['monitor']})
calorimetry = call('POST', '/reactivity/calorimetry/interpret', {'dataset': {'heatFlowW': 85, 'durationS': 900, 'massKg': 2.5}, 'vesselHeatCapacityCorrection': True})
reactivity_screen = call('POST', '/reactivity/screening', {'materials': ['ammonia', 'chlorine'], 'contaminants': ['water'], 'processConditions': {'temperatureC': 170}})
reactivity_control = call('POST', '/reactivity/control', {'hazardSummary': {'reactiveHazardPresent': True}, 'controlPreferences': ['inerting', 'temperature_control']})
device_select = call('POST', '/relief/devices/select', {'serviceType': 'vapor', 'cyclingExpected': True, 'allowableBackpressure': 0.08})
relief_system = call('POST', '/relief/system/analyze', {'scenario': 'external fire case', 'vessel': {'inventoryMassKg': 4200}})
effluent = call('POST', '/relief/effluent-handling/select', {'relievedMaterialState': 'vapor', 'toxic': False, 'flammable': True})
relief_liquid = call('POST', '/relief/sizing/liquid', {'requiredMassRate': 12, 'liquidDensity': 820, 'setPressure': 1_200_000, 'backpressure': 101_325})
relief_gas = call('POST', '/relief/sizing/gas-vapor', {'requiredMassRate': 6.5, 'temperatureK': 360, 'molecularWeight': 44.1, 'heatCapacityRatio': 1.13, 'setPressure': 1_500_000, 'backpressure': 101_325})
relief_two_phase = call('POST', '/relief/sizing/two-phase', {'requiredMassRate': 8.2, 'quality': 0.35})
deflag_vent = call('POST', '/relief/sizing/deflagration-vent', {'enclosureVolume': 250, 'reducedPressureTarget': 0.2, 'pmax': 8.0, 'kstOrKg': 180})
external_fire_relief = call('POST', '/relief/sizing/external-fire', {'wettedArea': 180})
thermal_expansion = call('POST', '/relief/sizing/thermal-expansion', {'blockedInVolume': 3.5, 'thermalExpansionCoefficient': 0.00095, 'temperatureRise': 40})

pprint({
    'purge_cycles': purge['cyclesRequired'],
    'static_risk_level': static_risk['riskLevel'],
    'area_classification': area['classification'],
    'fire_protection_gaps': fire_protection['gaps'],
    'reactivity_hazard_present': reactivity_screen['reactiveHazardPresent'],
    'relief_candidates': device_select['candidates'],
    'required_relief_load': relief_system['requiredReliefLoad'],
    'effluent_option': effluent['recommendedOption'],
    'liquid_relief_area': relief_liquid['requiredArea'],
    'gas_relief_area': relief_gas['requiredArea'],
    'two_phase_relief_area': relief_two_phase['requiredArea'],
    'deflagration_vent_area': deflag_vent['requiredVentArea'],
    'external_fire_relief_area': external_fire_relief['requiredArea'],
    'thermal_expansion_area': thermal_expansion['requiredArea'],
})


## 9) Hazard-Evaluation Workflows

In [ ]:
checklist = call('POST', '/hazard-evaluation/checklist', {'processId': 'NH3 storage', 'checklistItems': ['Isolation valves labeled', 'Emergency shower accessible']})
safety_review = call('POST', '/hazard-evaluation/safety-review', {'processId': 'NH3 storage', 'scope': 'MOC after relief upgrade'})
inherent = call('POST', '/hazard-evaluation/inherent-safety-review', {'processId': 'NH3 storage', 'strategies': ['minimize inventory', 'substitute hazardous reagent']})
pha = call('POST', '/hazard-evaluation/preliminary-hazard-analysis', {'processId': 'NH3 transfer', 'hazards': ['toxic release', 'jet fire'], 'causes': ['hose rupture', 'valve failure']})
ranking = call('POST', '/hazard-evaluation/relative-ranking', {'factors': {'inventory': 7, 'toxicity': 8, 'congestion': 5}})
hazop = call('POST', '/hazard-evaluation/hazop', {'nodes': ['Compressor discharge', 'Tank inlet'], 'guidewords': ['more pressure', 'reverse flow']})
fmea = call('POST', '/hazard-evaluation/fmea', {'equipmentItems': ['Transfer pump P-101', 'Relief valve PSV-12']})
what_if = call('POST', '/hazard-evaluation/what-if', {'processId': 'Loading bay', 'prompts': ['What if emergency shutdown fails?']})
what_if_checklist = call('POST', '/hazard-evaluation/what-if-checklist', {'processId': 'Loading bay', 'prompts': ['What if hose disconnects?'], 'checklistItems': ['operator present', 'wheel chocks in place']})
info_validation = call('POST', '/hazard-evaluation/information-requirements/validate', {
    'chemicals': {'flammability': True, 'toxicity': True, 'reactivity': True, 'physical_properties': True},
    'equipment': {'piping': 'complete'},
    'procedures': {'startup': 'draft'},
    'conditions': {'temperature': 25, 'pressure': 5, 'flow': 100},
})

print('Checklist findings:', len(checklist['findings']))
print('Safety review findings:', len(safety_review['findings']))
print('Inherent safety findings:', len(inherent['findings']))
print('PHA findings:', len(pha['findings']))
print('Relative ranking:', ranking)
print('HAZOP findings:', len(hazop['findings']))
print('FMEA findings:', len(fmea['findings']))
print('What-if findings:', len(what_if['findings']))
print('What-if/checklist findings:', len(what_if_checklist['findings']))
print('Information complete?:', info_validation['complete'])


## 10) Visualization, GIS, Sign Intelligence, And Integration Surfaces

In [ ]:
visual = call('POST', '/visualization/solve', {
    'layer_type': 'risk_contours',
    'inputs': {
        'scenario_type': 'fire',
        'source': {'latitude': 52.52, 'longitude': 13.405, 'label': 'LPG sphere'},
        'zones': [
            {'label': '12.5 kW/m2', 'threshold': 12.5, 'unit': 'kW/m2', 'radius_m': 140},
            {'label': '4 kW/m2', 'threshold': 4.0, 'unit': 'kW/m2', 'radius_m': 260},
        ],
    },
})
sign = call('POST', '/signs/analyze', {
    'observed_text': 'Danger Flammable Gas Pipeline',
    'site_context': 'High pressure line crossing',
    'stability_class': 'D',
    'wind_speed_m_s': 3.0,
})
gis_scenario = call('POST', '/gis/scenarios/evaluate', {
    'scenario_type': 'fire',
    'model_id': 'fire.point_source_heat_flux',
    'source': {'latitude': 52.52, 'longitude': 13.405, 'label': 'NH3 manifold'},
    'receptors': [
        {'id': 'R1', 'latitude': 52.521, 'longitude': 13.407, 'label': 'Control room'},
        {'id': 'R2', 'latitude': 52.519, 'longitude': 13.402, 'label': 'Gatehouse'},
    ],
    'inputs': {'burning_rate_kg_s': 2.8, 'heat_of_combustion_kj_kg': 46000},
})
impact = call('POST', '/gis/impact-zones', {
    'scenario_type': 'leak',
    'source': {'latitude': 52.52, 'longitude': 13.405, 'label': 'NH3 manifold'},
    'asset': {'line_pressure_kpa': 6500, 'mass_flow_kg_s': 1.8, 'gas_temperature_c': 18, 'leak_duration_s': 120, 'stability_class': 'D'},
    'criteria': [{'label': 'IDLH screening', 'threshold': 0.02, 'unit': 'kg/m^3'}],
})
catalog = call('GET', '/service-catalog')

print('Visualization layer type:', visual['layer_type'])
print('Sign recommended models:')
pprint(sign['recommended_models'])
print('GIS receptors evaluated:', len(gis_scenario['receptors']))
print('Impact zones generated:', len(impact['zones']))
print('Service-catalog entries:', len(catalog if isinstance(catalog, list) else catalog.get('services', [])))


## 11) Runtime Surface Quick Reference

These surfaces use the same API model chain shown above:

- **Jupyter**: this notebook
- **Browser-local**: `docs/app.html`
- **Docker**: `docker build -t deepsafety .` then `docker run --rm -p 8000:8000 deepsafety`
- **MCP**: `deepsafety-mcp` tool workflows
- **Python client**: `from deepsafety import DeepSafetyClient`
